# 📈 Previsão de Vendas Diárias em FMCG (2022–2024) – Machine Learning

## 📋 Introdução
Este projeto analisa o conjunto de dados **FMCG Daily Sales (2022–2024)** disponível no Kaggle, 
que simula um cenário realista de vendas diárias de produtos lácteos (leite e iogurte) 
na Polônia, considerando múltiplos **SKUs**, **canais de venda**, **regiões**, 
**promoções**, **preços** e **níveis de estoque**.

O dataset bruto contém **mais de 1 milhão de registros**, com variáveis como:
`date`, `sku`, `brand`, `channel`, `region`, `price`, `promotion`, 
`units_sold` e `stock_on_hand`.

---

## 🎯 Objetivo de Negócio
No setor de **FMCG (Fast-Moving Consumer Goods)**, erros de previsão de demanda 
podem resultar em **ruptura de estoque** ou **excesso de inventário**, ambos com 
impacto direto em custos e nível de serviço.

Este projeto tem como objetivo principal **prever a demanda diária** de um produto 
de alto volume, fornecendo subsídios para **planejamento de estoque** e 
**tomada de decisão operacional**.

---

## 🎯 Objetivos Técnicos
Os objetivos técnicos do projeto são:

1. **Bibliotecasa utilizadas**
2. **Contexto e carregamento dos dados**
3. **Análise Exploratória de Dados (EDA)** para identificar padrões de tendência, 
   sazonalidade e o impacto de promoções nas vendas.
4. **Modelagem de Séries Temporais**, utilizando:
   - Modelos clássicos (Prophet) - baseline;
   - Modelos de Machine Learning (XGBoost) com engenharia de atributos 
     baseada em *lags* e janelas móveis (*rolling features*).
5. **Avaliação dos modelos** por meio de validação cruzada temporal e métricas 
   como MAE e RMSE.
6. **Geração de insights de negócio**, incluindo previsão de vendas para os 
   próximos 30 dias com foco em suporte à gestão de estoque.

---

## 📌 Escopo do Projeto
- Análise focada em **um SKU de alto volume**, garantindo melhor relação sinal-ruído.
- Horizonte de previsão de **30 dias**.
- O projeto **não aborda**:
  - Otimização de preços;
  - Modelagem hierárquica multi-SKU;
  - Previsão multivariada por região ou canal simultaneamente.


# 1. BIBLIOTECAS:

In [140]:
%pip install prophet --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# MANIPULAÇÃO DE DADOS
import pandas as pd
import numpy as np

# VISUALIZAÇÃO DE DADOS
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# MODELOS
from prophet import Prophet
from xgboost import XGBRegressor
from xgboost import plot_importance

# MÉTRICAS
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 2. DATASET:

In [159]:
# CARREGAR O DATASET
df = pd.read_csv('FMCG_2022_2024.csv') 

In [163]:
# 5 PRIMEIRAS LINHAS DO DATASET:
df.head()

,date,sku,brand,segment,category,channel,region,pack_type,price_unit,promotion_flag,delivery_days,stock_available,delivered_qty,units_sold
0,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-Central,Multipack,2.38,0,1,141,128,9
1,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-North,Single,1.55,1,3,0,129,0
2,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-South,Carton,4.00,0,5,118,161,8
3,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-Central,Single,5.16,0,2,81,114,7
4,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-North,Single,7.66,0,4,148,204,12


In [162]:
# INFORMAÇÕES DO DATASET:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190757 entries, 0 to 190756
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   date             190757 non-null  object 
 1   sku              190757 non-null  object 
 2   brand            190757 non-null  object 
 3   segment          190757 non-null  object 
 4   category         190757 non-null  object 
 5   channel          190757 non-null  object 
 6   region           190757 non-null  object 
 7   pack_type        190757 non-null  object 
 8   price_unit       190757 non-null  float64
 9   promotion_flag   190757 non-null  int64  
 10  delivery_days    190757 non-null  int64  
 11  stock_available  190757 non-null  int64  
 12  delivered_qty    190757 non-null  int64  
 13  units_sold       190757 non-null  int64  
dtypes: float64(1), int64(5), object(8)
memory usage: 20.4+ MB


In [165]:
# NULOS NO DATASET:
df.isnull().sum()

date               0
sku                0
brand              0
segment            0
category           0
channel            0
region             0
pack_type          0
price_unit         0
promotion_flag     0
delivery_days      0
stock_available    0
delivered_qty      0
units_sold         0
dtype: int64

In [150]:
# DUPLICADOS NO DATASET:
print(f'Duplicados no dataset: {df.duplicated().sum()}')

Duplicados no dataset: 0


In [ ]:
# CONVERTENDO A COLUNA 'date' PARA DATETIME:
df['date'] = pd.to_datetime(df['date'])

In [157]:
# VERIFICANDO A CONVERSÃO:
df['date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 190757 entries, 0 to 190756
Series name: date
Non-Null Count   Dtype         
--------------   -----         
190757 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 1.5 MB


# 3. ANÁLISE EXPLORATÓRIA DE DADOS (EDA):

In [168]:
# SHAPE DATASET:
print(f"Shape do dataset: {df.shape}")
print(f"   • {df.shape[0]} registros")
print(f"   • {df.shape[1]} colunas")

Shape do dataset: (190757, 14)
   • 190757 registros
   • 14 colunas


In [169]:
# INTERVALO DE DATAS:
print('Data mínima:', df['date'].min())
print('Data máxima:', df['date'].max())

# DATAS ÚNICAS:
print('Dias únicos:', df['date'].nunique())

Data mínima: 2022-01-21
Data máxima: 2024-12-31
Dias únicos: 1076


In [ ]:
# QUANTIDADE DE VENDAS POR ANO:
vendas_ano = df.groupby('year')['units_sold'].sum()
print('Vendas por Ano:\n', vendas_ano)

Vendas por Ano:
 year
2022     604945
2023    1605008
2024    1589871
Name: units_sold, dtype: int64


In [175]:
# MÉDIA VENDAS DIÁRIAS TOTAIS:
total_vendas = df['units_sold'].sum()
print(f'Total de vendas: {total_vendas} unidades')

dias = df['date'].nunique()
print(f'Total de dias: {dias} dias')

print(f'Média diária: {total_vendas/dias:.0f} unidades')

Total de vendas: 3799824 unidades
Total de dias: 1076 dias
Média diária: 3531 unidades


In [181]:
# VENDAS DIÁRIAS COM MÉDIA MÓVEL DE 30 DIAS:
vendas_diarias = df.groupby('date')['units_sold'].sum().reset_index()

vendas_diarias['rolling_30d'] = (
    vendas_diarias['units_sold']
    .rolling(window=30)
    .mean()
)

In [185]:
fig = px.line(
    vendas_diarias,
    x='date',
    y=['units_sold', 'rolling_30d'],
    title='Vendas Totais Diárias (2022–2024)',
    labels={'value': 'Unidades Vendidas', 'variable': 'Série'}
)
fig.show()

In [193]:
# TOP SKUs E DISTRIBUIÇÃO CHANNEL:
channel_porcentagem = df['channel'].value_counts(normalize=True) * 100

print('\nCanais %:\n', channel_porcentagem.round(1).astype(str) + '%')
print('\nTop SKUs:\n', df['sku'].value_counts().head())
print('\nDistribuição Channel:\n', df['channel'].value_counts())


Canais %:
 channel
Retail        33.4%
E-commerce    33.4%
Discount      33.3%
Name: proportion, dtype: object

Top SKUs:
 sku
MI-006    8221
MI-026    8216
YO-029    7909
YO-005    7893
YO-012    7719
Name: count, dtype: int64

Distribuição Channel:
 channel
Retail        63688
E-commerce    63619
Discount      63450
Name: count, dtype: int64


In [213]:
# PROMOÇÃO E STOCKOUTS:
promocao_efeito = df.groupby('promotion_flag')['units_sold'].agg(['mean','count']).round(3)
print('Efeito da promoção\n', promocao_efeito)


stock_zero = (
    df[df['stock_available'] == 0]['date']
    .nunique()
)
total_dias = df['date'].nunique()
stockout_percentual = stock_zero / total_dias * 100
print(f'\nDias de Stockout: {stock_zero} dias ({stockout_percentual:.1f}%)')

Efeito da promoção
                   mean   count
promotion_flag                
0               17.440  162296
1               34.061   28461

Dias de Stockout: 991 dias (92.1%)


### Promoção e Stockouts

Nesta etapa, analisamos o impacto das promoções sobre o volume médio de vendas, bem como a frequência de eventos de ruptura de estoque (stockouts) ao longo do período analisado.

Inicialmente, comparamos a média de unidades vendidas entre períodos com e sem promoção.

O Stockout de 991 dias (92,1%) responde a seguinte pergunta: 'Em quantos dias existiu pelo menos um registro (SKU–canal–região) com estoque zero?'. Esse comportamento é esperado em bases de dados transacionais com múltiplos SKUs, canais e regiões, uma vez que a ocorrência de ao menos um evento de ruptura em um determinado dia não implica indisponibilidade generalizada de produtos.

### Promo Effect
Os resultados indicam que períodos com promoção apresentam um volume médio de vendas significativamente superior:

- Sem promoção (`promotion_flag = 0`): média de **17,4 unidades vendidas**
- Com promoção (`promotion_flag = 1`): média de **34,1 unidades vendidas**

Além disso, observa-se que aproximadamente **15% das observações** do dataset ocorreram sob algum tipo de promoção, enquanto a maior parte das vendas aconteceu em períodos sem incentivo promocional.

In [191]:
# MEDIDA DO IMPACTO DA PROMOÇÃO:
promocao_vendas = df.groupby('promotion_flag')['units_sold'].mean()
promocao_incremento = ((promocao_vendas[1] - promocao_vendas[0])/promocao_vendas[0]*100).round(0)
print(f'Incremento da promoção: +{promocao_incremento}% ({promocao_vendas[0]:.0f}→{promocao_vendas[1]:.0f})')

Incremento da promoção: +95.0% (17→34)


### Impacto das Promoções nas Vendas (Promo Lift)

Para quantificar o impacto das promoções sobre o volume médio de vendas, foi calculado o *promo lift*, que mede a variação percentual da média de unidades vendidas entre períodos com e sem promoção.

Os resultados indicam um *promo lift* de aproximadamente **+95%**, conforme descrito abaixo:

- Média sem promoção: **17 unidades**
- Média com promoção: **34 unidades**

Isso sugere que a presença de promoções está associada a um aumento de **duas vezes** no volume de vendas, reforçando a relevância dessa variável como um importante fator explicativo para modelos de previsão de demanda.


In [206]:
# SAZONALIDADE MÊS/ANO:
vendas_mensais = df.groupby(['year','month'])['units_sold'].sum().reset_index()
fig = px.density_heatmap(vendas_mensais, x='month', y='year', z='units_sold',
                        title='Sazonalidade Mês/Ano', color_continuous_scale='YlGnBu')

fig.update_layout(
    title_x=0.5,  # CENTRALIZAR TÍTULO
    xaxis_title='Mês', # TÍTULO DO EIXO X
    yaxis_title='Ano', # TÍTULO DO EIXO Y
    coloraxis_colorbar_title='Unidades Vendidas') # LEGENDA DA COR

fig.show()

- **Vendas Totais:** ~ 3.5K unidades/dia
- **Channel:** Retail 33.3% | E-commerce 33.3% | Discount 33.3%
- **Efeito das promoções:** +95% lift vendas (17.440 → 34.061 unidades/dia)
- **Dias de Stockouts:** 991 dias (92.01% total)
- **Sazonalidade:** Picos de venda no meio do ano

# 4. PRÉ-PROCESSAMENTO:

## Pré-processamento dos Dados

Após a análise exploratória global, o projeto passa a focar em um único SKU de alto
volume de vendas (MI-006), permitindo uma modelagem de séries temporais mais robusta
e com melhor relação sinal-ruído.

In [221]:
# ESCOLHER A SKU 'MI-006' (MAIOR NÉMURO DE VENDAS) PARA MODELAGEM:
# ESCOLHER A SKU 'MI-006' E AGREGAR POR DATA (1 LINHA = 1 DIA)
df_mi = (
    df[df['sku'] == 'MI-006']
    .groupby('date')
    .agg({
        'units_sold': 'sum',
        'promotion_flag': 'max'
    })
    .sort_index()
)

# GARANTIR TIPO NUMÉRICO
df_mi['units_sold'] = df_mi['units_sold'].astype(float)

In [222]:
# FEATURE ENGINEERING:
df_mi['lag_1d'] = df_mi['units_sold'].shift(1) # LAG DE 1 DIA - VALOR DO DIA ANTERIOR 
df_mi['lag_7d'] = df_mi['units_sold'].shift(7) # LAG DE 7 DIAS - VALOR DA SEMANA ANTERIOR
df_mi['roll_mean_7'] = df_mi['units_sold'].rolling(7).mean() # MÉDIA MÓVEL 7 DIAS - TENDÊNCIA LOCAL
df_mi['is_promo'] = (df_mi['promotion_flag']==1).astype(int) # FLAG DE PROMOÇÃO (1=SIM, 0=NÃO)
df_mi['year'] = df_mi.index.year # EXTRAIR ANO DA DATA - ÚTIL PARA SAZONALIDADE
df_mi['month'] = df_mi.index.month # EXTRAIR MÊS DA DATA - ÚTIL PARA SAZONALIDADE

In [223]:
print('Shape de df_mi:', df_mi.shape)

Shape de df_mi: (1076, 8)


In [224]:
df_mi[['units_sold','lag_7d','roll_mean_7','is_promo']].dropna()
# shift (1) e shift (7) CRIAM NULOS NO INÍCIO DO DATASET DEVIDO AO LAG DE 1 E 7 DIAS, RESPECTIVAMENTE.

,units_sold,lag_7d,roll_mean_7,is_promo
date,,,,
2022-01-28,132.0,85.0,100.000000,1
2022-01-29,98.0,119.0,97.000000,1
2022-01-30,78.0,122.0,90.714286,1
2022-01-31,115.0,76.0,96.285714,1
2022-02-01,77.0,76.0,96.428571,1
...,...,...,...,...
2024-12-27,114.0,129.0,100.571429,1
2024-12-28,103.0,105.0,100.285714,1
2024-12-29,124.0,113.0,101.857143,1


## 🔧 Pré-processamento e Engenharia de Atributos

Após a seleção do SKU MI-006, os dados foram ordenados temporalmente e utilizados para
a criação de atributos baseados em séries temporais, incluindo valores defasados
(*lags*), médias móveis e variáveis de calendário.

Essas transformações permitem que modelos supervisionados capturem dependências
temporais, efeitos de promoção e padrões sazonais presentes na série.

O Prophet foi utilizado como modelo baseline por sua capacidade nativa de capturar
tendência e sazonalidade, exigindo mínimo pré-processamento. Em uma etapa posterior,
modelos de machine learning (XGBoost) foram explorados, incorporando engenharia de
atributos explícita, incluindo representações cíclicas de variáveis temporais.


# 5. MODELAGEM:

In [225]:
# SPLIT DE TESTE E TREINO - NÃO PODE SER ALEATÓRIO
# 80% TREINO: SETEMBRO 2022 - SETEMBRO 2024
# 20% TESTE: OUTUBRO 2024 - DEZEMBRO 2024
data_separacao = '2024-10-01'

df_train = df_mi.loc[df_mi.index < data_separacao].dropna() # TODAS AS DATAS ATÉ 30 SETEMBRO 2024
df_test  = df_mi.loc[df_mi.index >= data_separacao].dropna() # TODAS AS DATAS A PARTIR DE 01 OUTUBRO 2024

## 5.1. PROPHET:

In [226]:
# PREPARAR DATAFRAME PARA PROPHET:

# RENAME COLUNAS PARA O FORMATO DO PROPHET (ds E y):
# date → ds
# units_sold → y
# date VIRA COLUNA NOVAMENTE
df_prophet = df_mi[['units_sold']].reset_index().rename(columns={'date':'ds', 'units_sold':'y'})

# REGRESSOR EXTERNO DE PROMOÇÃO:
df_prophet['promo'] = df_mi['promotion_flag'].values

In [227]:
# TREINO
train_prophet = df_prophet[df_prophet['ds'] < data_separacao]

# TESTE
test_prophet = df_prophet[df_prophet['ds'] >= data_separacao]

print('Treino:', train_prophet.shape)
print('Teste:', test_prophet.shape)

Treino: (984, 3)
Teste: (92, 3)


In [228]:
# INICIALIZAR O MODELO PROPHET (BASELINE)
modelo_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
modelo_prophet.add_regressor('promo')

# TREINAR O MODELO
modelo_prophet.fit(train_prophet)

20:13:44 - cmdstanpy - INFO - Chain [1] start processing
20:13:45 - cmdstanpy - INFO - Chain [1] done processing


In [229]:
# DATAFRAME FUTURO PARA O PERÍODO DE TESTE
test_futuro = test_prophet[['ds', 'promo']].copy()

# GERAR PREVISÕES
test_previsao = modelo_prophet.predict(test_futuro)

test_previsao[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head()

,ds,yhat,yhat_lower,yhat_upper
0,2024-10-01,107.255958,72.971376,143.669275
1,2024-10-02,106.235895,70.338505,142.523945
2,2024-10-03,83.668042,51.000574,115.074078
3,2024-10-04,104.541899,70.378422,139.542010
4,2024-10-05,108.367858,75.193708,144.184858


### 5.1.1. MÉTRICAS DO MODELO PROPHET:

In [230]:
# REAL X PREVISTO:
# RESET INDEX PARA COMPARAÇÃO
df_real = df_mi.reset_index()[['date', 'units_sold']]
df_real.columns = ['ds', 'y']

In [231]:
# FILTRAR APENAS O PERÍODO DE TESTE
df_real_test = df_real[df_real['ds'].isin(test_previsao['ds'])]

In [232]:
# JUNTAR REAL E PREVISTO
df_avaliacao = df_real_test.merge(
    test_previsao[['ds', 'yhat']],
    on='ds',
    how='inner'
)

In [233]:
df_avaliacao.head()

,ds,y,yhat
0,2024-10-01,100.0,107.255958
1,2024-10-02,113.0,106.235895
2,2024-10-03,138.0,83.668042
3,2024-10-04,61.0,104.541899
4,2024-10-05,96.0,108.367858


In [234]:
# MÉTRICAS DE AVALIAÇÃO:
mae_prophet = mean_absolute_error(df_avaliacao['y'], df_avaliacao['yhat'])
rmse_prophet = np.sqrt(mean_squared_error(df_avaliacao['y'], df_avaliacao['yhat']))

print(f"MAE: {mae_prophet:.2f}")
print(f"RMSE: {rmse_prophet:.2f}")

MAE: 17.57
RMSE: 22.59


In [240]:
# VISUALIZAR PREVISÕES VS REAL COM INTERVALO DE CONFIANÇA:
fig = go.Figure()

# INTERVALO DE CONFIANÇA
fig.add_trace(
    go.Scatter(
        x=test_previsao['ds'],
        y=test_previsao['yhat_upper'], # LIMITE SUPERIOR
        mode='lines',
        line=dict(width=0),
        showlegend=False # NÃO MOSTRAR NA LEGENDA
    )
)

fig.add_trace(
    go.Scatter(
        x=test_previsao['ds'],
        y=test_previsao['yhat_lower'], # LIMITE INFERIOR
        mode='lines',
        fill='tonexty', # PREENCHER A ÁREA ENTRE AS LINHAS
        fillcolor='rgba(0, 100, 80, 0.2)', # COR DO PREENCHIMENTO
        line=dict(width=0),
        name='Intervalo de Confiança'
    )
)

# PREVISÃO
fig.add_trace(
    go.Scatter(
        x=test_previsao['ds'],
        y=test_previsao['yhat'],
        mode='lines', # LINHA
        name='Previsão (Prophet)',
        line=dict(dash='dash') # LINHA TRACEJADA
    )
)

# REAL
fig.add_trace(
    go.Scatter(
        x=df_avaliacao['ds'],
        y=df_avaliacao['y'],
        mode='lines', # LINHA
        name='Vendas Reais'
    )
)

fig.update_layout(
    title='Vendas Reais vs Previsão com Intervalo de Confiança',
    xaxis_title='Data',
    yaxis_title='Unidades Vendidas',
    hovermode='x unified' # MOSTRAR VALORES DE TODAS AS SÉRIES NO MESMO PONTO DO EIXO X
)

fig.show()

#### Métricas do Modelo Prophet

Para avaliar o desempenho do modelo **Prophet**, utilizamos métricas de erro amplamente empregadas em problemas de **previsão de séries temporais**, comparando os valores previstos com as vendas reais no período de teste (outubro/2024 a dezembro/2024).

As métricas adotadas foram:

- **MAE (Mean Absolute Error)**: mede o erro médio absoluto entre as previsões e os valores reais, sendo interpretável na mesma unidade da variável alvo (unidades vendidas).
- **RMSE (Root Mean Squared Error)**: penaliza erros maiores de forma mais severa, sendo sensível a grandes desvios pontuais.

#### Resultados obtidos:

- **MAE:** 17,57 unidades  
- **RMSE:** 22,59 unidades  

Esses valores indicam que, em média, o modelo Prophet apresenta um erro absoluto de aproximadamente **18 unidades vendidas por dia**, com maior penalização em dias onde ocorreram variações mais abruptas de demanda.

#### Análise qualitativa do desempenho:

A análise visual entre vendas reais e previsões mostra que o Prophet:

- Captura adequadamente a **tendência geral** da série;
- Modela bem os **padrões sazonais semanais e anuais**;
- Apresenta maior dificuldade em responder a **picos e quedas bruscas**, comuns em períodos promocionais ou eventos específicos.

Esse comportamento é esperado, uma vez que o Prophet é um modelo **aditivo e global**, projetado para capturar estrutura temporal de forma robusta, porém com menor sensibilidade a efeitos locais e interações complexas entre variáveis.

#### Conclusão:

O modelo Prophet cumpre bem seu papel como **baseline**, fornecendo uma referência sólida de desempenho.  
No entanto, suas limitações na captura de efeitos não lineares e impactos locais reforçam a motivação para a aplicação de um modelo mais flexível, como o **XGBoost**, que será explorado na próxima etapa do projeto.

## 5.2. XGBOOST:

In [243]:
# COPIAR DATAFRAME PARA EVITAR EFEITOS COLATERAIS
df_xgb = df_mi.copy()

In [244]:
# DEFINIR FEATURES E TARGET
features = [
    'lag_1d',
    'lag_7d',
    'roll_mean_7',
    'is_promo',
    'month',
    'year'
]

target = 'units_sold'

In [245]:
data_separacao = '2024-10-01'

df_train_xgb = df_xgb.loc[df_xgb.index < data_separacao].dropna()
df_test_xgb  = df_xgb.loc[df_xgb.index >= data_separacao].dropna()

In [ ]:
# PREPARAR MATRIZES DE TREINO E TESTE:
X_train = df_train_xgb[features]
y_train = df_train_xgb[target]

X_test  = df_test_xgb[features]
y_test  = df_test_xgb[target]

In [ ]:
# INICIALIZAR E TREINAR O MODELO XGBOOST:
modelo_xgb = XGBRegressor(
    n_estimators=300, # NÚMERO DE ÁRVORES
    learning_rate=0.05, # TAXA DE APRENDIZADO
    max_depth=5, # PROFUNDIDADE MÁXIMA DAS ÁRVORES
    subsample=0.8, 
    colsample_bytree=0.8, # SUBSAMPLE DE COLUNAS POR ÁRVORE
    random_state=42
)

In [248]:
# TREINAR O MODELO
modelo_xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# GERAR PREDIÇÕES:
y_pred_xgb = modelo_xgb.predict(X_test)

In [ ]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"MAE XGBoost: {mae_xgb:.2f}")
print(f"RMSE XGBoost: {rmse_xgb:.2f}")

In [250]:
df_comparacao_xgb = pd.DataFrame({
    'data': X_test.index,
    'real': y_test.values,
    'predito_xgb': y_pred_xgb
})

df_comparacao_xgb.head()

,data,real,predito_xgb
0,2024-10-01,100.0,116.852158
1,2024-10-02,113.0,123.317337
2,2024-10-03,138.0,101.882629
3,2024-10-04,61.0,105.064751
4,2024-10-05,96.0,125.261497


### Comparação entre Valores Reais e Previstos — XGBoost

Foi construída uma base de comparação contendo as vendas reais e as previsões geradas pelo modelo XGBoost no período de teste.  
Essa análise permite avaliar qualitativamente o comportamento do modelo, especialmente sua capacidade de responder a variações diárias e picos de demanda.


### 5.2.1. MÉTRICAS:

In [251]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"MAE (XGBoost): {mae_xgb:.2f}")
print(f"RMSE (XGBoost): {rmse_xgb:.2f}")

MAE (XGBoost): 17.02
RMSE (XGBoost): 21.04


In [252]:
print(f'mae (PROPHET): {mae_prophet:.2f}')
print(f'mae (XGBOOST): {mae_xgb:.2f}')
print(f'rmse (PROPHET): {rmse_prophet:.2f}')
print(f'rmse (XGBOOST): {rmse_xgb:.2f}')    

mae (PROPHET): 17.57
mae (XGBOOST): 17.02
rmse (PROPHET): 22.59
rmse (XGBOOST): 21.04


#### Comparação de Desempenho entre Modelos

Os modelos Prophet e XGBoost foram avaliados no conjunto de teste utilizando as métricas MAE e RMSE.  
O XGBoost apresentou melhor desempenho em ambas as métricas, indicando maior capacidade de capturar variações diárias e picos de demanda, enquanto o Prophet serviu como um baseline robusto e interpretável.


In [ ]:
# CALCULAR INTERVALO DE CONFIANÇA BASEADO NOS RESÍDUOS DO MODELO XGBOOST:
residuos = df_comparacao_xgb['real'] - df_comparacao_xgb['predito_xgb']
std_resid = residuos.std()

df_comparacao_xgb['lower'] = df_comparacao_xgb['predito_xgb'] - 1.96 * std_resid
df_comparacao_xgb['upper'] = df_comparacao_xgb['predito_xgb'] + 1.96 * std_resid

In [ ]:
# VISUALIZAR PREVISÕES VS REAL COM INTERVALO DE CONFIANÇA:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_comparacao_xgb['data'],
    y=df_comparacao_xgb['real'],
    mode='lines',
    name='Vendas Reais',
    line = dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=df_comparacao_xgb['data'],
    y=df_comparacao_xgb['predito_xgb'],
    mode='lines',
    name='Previsão XGBoost',
    line = dict(color='red', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=list(df_comparacao_xgb['data']) + list(df_comparacao_xgb['data'])[::-1],
    y=list(df_comparacao_xgb['upper']) + list(df_comparacao_xgb['lower'])[::-1],
    fill='toself',
    fillcolor='rgba(0,100,255,0.2)',
    line=dict(color='rgba(120,205,200,0)'),
    name='Intervalo de Confiança (95%)'
))

fig.update_layout(
    title='XGBoost — Previsão com Intervalo de Confiança (95%)',
    xaxis_title='Data',
    yaxis_title='Unidades Vendidas',
    title_x=0.5
)

fig.show()


In [280]:
# COMPARAR AS PREVISÕES DOS DOIS MODELOS (PROPHET E XGBOOST):
df_compare = df_comparacao_xgb.merge(
    df_avaliacao[['ds', 'yhat']],
    left_on='data',
    right_on='ds',
    how='left'
).rename(columns={'yhat': 'predito_prophet'})

df_compare = df_compare[['data', 'real', 'predito_prophet', 'predito_xgb']]

In [281]:
# VISUALIZAR COMPARAÇÃO DAS PREVISÕES:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_compare['data'],
    y=df_compare['real'],
    mode='lines',
    name='Vendas Reais',
    line=dict(color = 'blue',width=2)
))

fig.add_trace(go.Scatter(
    x=df_compare['data'],
    y=df_compare['predito_prophet'],
    mode='lines',
    name='Previsão Prophet',
    line=dict(color = 'red', dash='dot')
))

fig.add_trace(go.Scatter(
    x=df_compare['data'],
    y=df_compare['predito_xgb'],
    mode='lines',
    name='Previsão XGBoost',
    line=dict(color = 'green', dash='dash')
))

fig.update_layout(
    title='Vendas Reais vs Previsões — Prophet x XGBoost (SKU MI-006)',
    xaxis_title='Data',
    yaxis_title='Unidades Vendidas',
    title_x=0.5,
    template = 'plotly_white'
)

fig.show()


In [285]:
# SIMULACAO DE IMPACTO
buffer_seguranca = np.percentile(abs(residuos), 95)


- Buffer de Segurança = erro máximo esperado em 95% dos casos

- Tradução: “Se eu adicionar esse buffer ao estoque previsto, vou cobrir 95% das variações inesperadas da demanda.”

In [287]:
# CÁLCULO DO ESTOQUE RECOMENDADO:
# PREDITO + BUFFER DE SEGURANÇA
df_comparacao_xgb['estoque_recomendado'] = (
    df_comparacao_xgb['predito_xgb'] + buffer_seguranca
)

In [288]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_comparacao_xgb['data'],
    y=df_comparacao_xgb['real'],
    name='Demanda Real',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=df_comparacao_xgb['data'],
    y=df_comparacao_xgb['estoque_recomendado'],
    name='Estoque Recomendado',
    line=dict(color = 'red', dash='dash')
))

fig.update_layout(
    title='Simulação de Planejamento de Estoque — SKU MI-006',
    xaxis_title='Data',
    yaxis_title='Unidades',
    title_x=0.5
)

fig.show()


# 6. CONCLUSÕES E ETAPAS FUTURAS:

## Insights de Negócio

- O modelo Prophet apresentou previsões mais estáveis, funcionando como um baseline confiável para entendimento de tendência e sazonalidade da demanda.
- O XGBoost demonstrou maior capacidade de capturar variações diárias e picos de demanda, resultando em menor erro médio (MAE e RMSE).
- A incorporação de um buffer de segurança baseado na distribuição empírica dos erros permite mitigar riscos de ruptura de estoque sem superdimensionamento excessivo.
- A abordagem combinada de previsão pontual e análise de incerteza fornece suporte prático à tomada de decisão operacional e ao planejamento de estoque.

## Próximos Passos
- Avaliar modelos híbridos (Prophet + ML) ou ainda modelos de Deep Learning como LSTM
- Incluir variáveis externas adicionais (preço, campanhas)
- Automatizar re-treinamento e monitoramento de erro
- Expandir abordagem para múltiplas SKUs

## Observação
Este estudo foi conduzido para uma única SKU como prova de conceito, podendo ser estendido para um portfólio completo de produtos.